This notebook contains the relevant code to batch process a dataset, given an input folder of science frames and an input folder of dark frames. 

In [ ]:
# import statements
from astropy.io import fits
import numpy as np
import os
import filter
import glob

In [ ]:
# set filepaths to relevant folders
dark_filepath = ""
sci_filepath = ""
output_filepath = ""

In [ ]:
# set filter parameters
ds_saturation_limit = 15000
ds_transition_width = 250
apply_bandstop_filter = True
bandstop_threshold = 0.025
bandstop_zeroth_order_transmission_width = 0.05
bandstop_gaussian_kernel_radius = 2
apply_wiener_filter = True
wiener_weight = 1
wiener_power = 0.5

In [ ]:
# run the main batch processing code
## grab dark frames
dark_psd = filter.construct_psd_from_directory(dark_filepath)
master_dark = filter.average_dataset_from_directory(dark_filepath)

# Create output directory if it doesn't exist
if not os.path.exists(output_filepath):
    os.makedirs(output_filepath)
    print(f"Created output directory: {output_filepath}")

# Gather all fits files
file_patterns = ['*.fits', '*.fit']
file_list = []
for pattern in file_patterns:
    file_list.extend(glob.glob(os.path.join(sci_filepath, pattern)))

print(f"Found {len(file_list)} files to process.")

# Iterate and process through each file
for file_path in file_list:
    try:
        # Get the filename to construct the output path
        filename = os.path.basename(file_path)
        output_path = os.path.join(output_filepath, filename)

        # Open the FITS file
        with fits.open(file_path) as hdul:
            sci_image = np.array(hdul[0].data)
            header = hdul[0].header.copy()

            # perform dark subtraction
            sci_image = filter.smart_dark_sub(sci_image,
                                              master_dark,
                                              saturation_limit=ds_saturation_limit,
                                              transition_width=ds_transition_width)
            sci_image[sci_image<0] = 0

            # generate bandstop filter
            bandstop_filter_mask = filter.create_bandstop_filter(dark_psd,
                                                                 bandstop_threshold,
                                                                 bandstop_zeroth_order_transmission_width,
                                                                 bandstop_gaussian_kernel_radius)
            
            # generate wiener filter
            sci_ft = np.fft.fft2(sci_image)
            sci_psd = np.abs(sci_ft)**2
            wiener_filter_mask = filter.create_wiener_filter(sci_psd,
                                                             dark_psd,
                                                             wiener_weight,
                                                             wiener_power)
            
            # apply filters
            if apply_bandstop_filter:
                sci_ft = bandstop_filter_mask * sci_ft
            
            if apply_wiener_filter:
                sci_ft = wiener_filter_mask * sci_ft

            sci_image = np.abs(np.fft.ifft2(sci_ft))
                
            # Update header
            header['HISTORY'] = 'Applied filtering to remove noise'

            # Save to new location
            fits.writeto(output_path, sci_image, header, overwrite=True)
                
        print(f"Saved: {filename}")

    except Exception as e:
        print(f"Failed to process {file_path}: {e}")

print("Batch processing complete.")